In [1]:
%matplotlib qt
import mne

#mne.viz.set_3d_backend('pyvistaqt')
from mne.coreg import Coregistration
from mne.io import read_info


import numpy as np
#%matplotlib qt
import matplotlib
#matplotlib.use('qt5agg')  # Or any other backend you want to use

import matplotlib.pyplot as plt

import pandas as pd 
import os
from os.path import join as pathjoin
from pathlib import Path

import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from time import time

from autoreject import AutoReject

import glob

import psutil
import gc
from time import time

mne.set_log_level('INFO')

import json

from mne.channels import read_dig_polhemus_isotrak  # Función para leer archivos .pos

import re
# from mne.minimum_norm import apply_inverse, make_inverse_operator
#este codigo lo dejo comentado para acostumbrarme a su suso

import brainiak
from brainiak.isc import isc 

import statsmodels
from statsmodels.stats.multitest import multipletests

In [2]:
try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_path         → g:\MOUS_204\MOUS_visual\output_source\source_block

In [ ]:
subdirectorio

In [3]:
#subjects = subj[:9]

subj = []

# Recorremos cada subdirectorio en la carpeta base
for subdirectorio in general_datadir.iterdir():
    # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
    if modality == "visual":
        subject_prefix = 'sub-V1'
    elif modality == "auditory":
        subject_prefix = 'sub-A2'
        
    if subdirectorio.is_dir() and subdirectorio.name.startswith(subject_prefix):
        # Añadimos el nombre del sujeto a la lista
        subj.append(subdirectorio.name)

print(subj)
subjects= subj [:20]
subjects

['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 'sub-V1011', 'sub-V1012', 'sub-V1013', 'sub-V1015', 'sub-V1016', 'sub-V1017', 'sub-V1019', 'sub-V1020', 'sub-V1022', 'sub-V1024', 'sub-V1025', 'sub-V1026', 'sub-V1027', 'sub-V1028', 'sub-V1029', 'sub-V1030', 'sub-V1031', 'sub-V1032', 'sub-V1033', 'sub-V1034', 'sub-V1035', 'sub-V1036', 'sub-V1037', 'sub-V1038', 'sub-V1039', 'sub-V1040', 'sub-V1042', 'sub-V1044', 'sub-V1045', 'sub-V1046', 'sub-V1048', 'sub-V1049', 'sub-V1050', 'sub-V1052', 'sub-V1053', 'sub-V1054', 'sub-V1055', 'sub-V1057', 'sub-V1058', 'sub-V1059', 'sub-V1061', 'sub-V1062', 'sub-V1063', 'sub-V1064', 'sub-V1065', 'sub-V1066', 'sub-V1068', 'sub-V1069', 'sub-V1070', 'sub-V1071', 'sub-V1072', 'sub-V1073', 'sub-V1074', 'sub-V1075', 'sub-V1076', 'sub-V1077', 'sub-V1078', 'sub-V1079', 'sub-V1080', 'sub-V1081', 'sub-V1083', 'sub-V1084', 'sub-V1085', 'sub-V1086', 'sub-V1087', 'sub-V1088', 'sub-V1089'

['sub-V1001',
 'sub-V1002',
 'sub-V1003',
 'sub-V1004',
 'sub-V1005',
 'sub-V1006',
 'sub-V1007',
 'sub-V1008',
 'sub-V1009',
 'sub-V1010',
 'sub-V1011',
 'sub-V1012',
 'sub-V1013',
 'sub-V1015',
 'sub-V1016',
 'sub-V1017',
 'sub-V1019',
 'sub-V1020',
 'sub-V1022',
 'sub-V1024']

In [14]:
def print5(*args):
    print(*(f"{x:.5f}" if isinstance(x, float) else x for x in args))

channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]

channels_mag=channels_mag.tolist()
print5(channels_mag)
indice_channels_efectivos = channels[channels[f"canal_efectivo_{modality}"].notna()]["indice"]
indice_channels_efectivos=indice_channels_efectivos.tolist()
# del channels


['MLC11-4304', 'MLC12-4304', 'MLC13-4304', 'MLC14-4304', 'MLC15-4304', 'MLC16-4304', 'MLC17-4304', 'MLC21-4304', 'MLC22-4304', 'MLC23-4304', 'MLC24-4304', 'MLC25-4304', 'MLC31-4304', 'MLC32-4304', 'MLC41-4304', 'MLC42-4304', 'MLC51-4304', 'MLC52-4304', 'MLC53-4304', 'MLC54-4304', 'MLC55-4304', 'MLC61-4304', 'MLC62-4304', 'MLC63-4304', 'MLF11-4304', 'MLF12-4304', 'MLF13-4304', 'MLF14-4304', 'MLF21-4304', 'MLF22-4304', 'MLF23-4304', 'MLF24-4304', 'MLF25-4304', 'MLF31-4304', 'MLF32-4304', 'MLF33-4304', 'MLF34-4304', 'MLF35-4304', 'MLF41-4304', 'MLF42-4304', 'MLF43-4304', 'MLF44-4304', 'MLF45-4304', 'MLF46-4304', 'MLF51-4304', 'MLF52-4304', 'MLF53-4304', 'MLF54-4304', 'MLF55-4304', 'MLF56-4304', 'MLF61-4304', 'MLF63-4304', 'MLF64-4304', 'MLF65-4304', 'MLF66-4304', 'MLF67-4304', 'MLO11-4304', 'MLO12-4304', 'MLO13-4304', 'MLO14-4304', 'MLO21-4304', 'MLO22-4304', 'MLO23-4304', 'MLO24-4304', 'MLO31-4304', 'MLO32-4304', 'MLO33-4304', 'MLO34-4304', 'MLO41-4304', 'MLO42-4304', 'MLO43-4304', 'MLO4

In [ ]:
# subj="sub-A2004"
# evokeds_zinnen=mne.read_evokeds(evoked_path / f"{subj}_evoked_zinnen_block-ave.fif")
# evoked_zinnen=evokeds_zinnen[0]
# data_subj=evoked_zinnen.pick("mag", exclude="bads").data

# n_dipoles=data_subj.shape[0]
# n_times=data_subj.shape[1]
# # subjects=["sub-A2002", "sub-A2003"]
# n_subjects=len(subjects)
# array_zinnen=np.zeros((n_times,n_dipoles, n_subjects))
# array_woorden=array_zinnen

# del evokeds_zinnen, evoked_zinnen, data_subj

# channels = pd.read_csv(channels_structure_path / "channels_mag.csv")
# channels_mag=channels[channels["canal_efectivo"].notna()]["canal_efectivo"]
# channels_mag=channels_mag.tolist()
# print(channels_mag)
# del channels

In [11]:
array_zinnen=np.zeros((n_times,n_dipoles, n_subjects))

number_subjects=np.zeros(len(subjects))

for i in range(0,n_subjects):
    try:
        subj=subjects[i]
        
        evokeds_zinnen=mne.read_evokeds(evoked_path / f"{subj}_evoked_zinnen_block-ave.fif")
        ## evokeds_zinnen es una lista

        ##cojo el primer elemento, solo tengo una lista
        evoked_zinnen=evokeds_zinnen[0]
        ####stc comes as an array of (n_dipoles, n_times)
        #take only the meg channels
        data_subj=evoked_zinnen.pick("mag", exclude="bads").data
        n_dipoles=data_subj.shape[0]
        n_times=data_subj.shape[1]
        # del stc_zinnen_morphed
        ##as everything on an array is of same dtype,  number of subject)
        subj_number = int(re.findall(r'\d+', subj)[0])  # Encuentra los dígitos y convierte a entero

        # Transponer los datos (n_times x n_dipoles -> n_dipoles x n_times)
        data_subj_swapped = data_subj.T

        array_zinnen[:,:,i]=data_subj_swapped
            # number_subjects[i]=subj_number
            # del data_subj
    except:
        print(f"Error en {subj}")
        continue

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1001_evoked_zinnen_block-ave.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms (fix_zinnen)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1002_evoked_zinnen_block-ave.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms (fix_zinnen)
        0 CTF compensation matrices available
        nave = 22 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Error en sub-V1003
Error en sub-V1004
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1005_evoked_zinnen_block-ave.fif ...
    Found the data of interest:
        t =   

In [15]:
## ZINNEN ISCS

# subjects=["sub-A2002", "sub-A2003"]
n_subjects=len(subjects)
# del stc_zinnen_general
# del data_subj
# del subj
array_zinnen=np.zeros((n_times,n_dipoles, n_subjects))

number_subjects=np.zeros(len(subjects))

for i in range(0,n_subjects):
    try:
        subj=subjects[i]
        
        evokeds_zinnen=mne.read_evokeds(evoked_path / f"{subj}_evoked_zinnen_block-ave.fif")
        ## evokeds_zinnen es una lista

        ##cojo el primer elemento, solo tengo una lista
        evoked_zinnen=evokeds_zinnen[0]
        ####stc comes as an array of (n_dipoles, n_times)
        #take only the meg channels
        data_subj=evoked_zinnen.pick("mag", exclude="bads").data
        n_dipoles=data_subj.shape[0]
        n_times=data_subj.shape[1]
        # del stc_zinnen_morphed
        ##as everything on an array is of same dtype,  number of subject)
        subj_number = int(re.findall(r'\d+', subj)[0])  # Encuentra los dígitos y convierte a entero

        # Transponer los datos (n_times x n_dipoles -> n_dipoles x n_times)
        data_subj_swapped = data_subj.T

        array_zinnen[:,:,i]=data_subj_swapped
        # number_subjects[i]=subj_number
        # del data_subj
    except:
        print(f"Error en {subj}")
        continue


#number_subjects_expanded = np.broadcast_to(number_subjects, (n_times,n_dipoles, n_subjects))

#array_zinnen[:, :, :] = number_subjects_expanded




## brainiak need an array like (n_TRs x n_voxels x n_subjects)
#so i have to create an array
#calculus of ISC correlation

iscs_zinnen= isc(data=array_zinnen, pairwise=False, summary_statistic=None, tolerate_nans=True)

iscs_statistics_zinnen=brainiak.isc.compute_summary_statistic(iscs_zinnen, summary_statistic='mean', axis=0)
iscs_statistics_zinnen.shape


##bootstrap zinnen gives you the isc values, the confidence intervals, the p values and the distribution of the isc values
## so  isc_zinnen=iscs_bootstrap_zinnen [0]

iscs_bootstrap_zinnen, ci_zinnen,p_zinnen, distribution_zinnen= brainiak.isc.bootstrap_isc(iscs_zinnen, 
pairwise=False, summary_statistic='median', n_bootstraps=1000, 
ci_percentile=95, side='right', random_state=None)


print(f"iscs_statistics_zinnen, {iscs_statistics_zinnen}")
print(f"iscs_bootstrap_zinnen: {iscs_zinnen}")
print(f"ci_zinnen: {ci_zinnen}")
print(f"p_zinnen: {p_zinnen}")
print(f"distribution_zinnen: {distribution_zinnen}")
print(f"iscs_statistics_zinnen: {iscs_statistics_zinnen}")

threshold = 0.05

# Encontrar los índices (canales) donde p_zinnen[0] es menor que 0.05
significant_channels_zinnen = np.where(p_zinnen < threshold)[0]

# Contar cuántos canales cumplen la condición
num_significant_channels_zinnen = len(significant_channels_zinnen)

# Imprimir los canales significativos
print("Canales significativos zinnen (p < 0.05):", significant_channels_zinnen)
print("Número total de canales significativos general:", num_significant_channels_zinnen)



# Aplicar la corrección de Benjamini-Hochberg (FDR)
_, p_adjusted_zinnen, _, _ = multipletests(p_zinnen, method='fdr_bh')
significant_channels_adjusted_zinnen = np.where(p_adjusted_zinnen < threshold)[0]

# Mostrar cuántos canales siguen siendo significativos
num_significant_channels_adjusted_zinnen = len(significant_channels_adjusted_zinnen)
print(f"Canales significativos zinnen después de FDR-BH: {significant_channels_adjusted_zinnen} de {len(channels_mag)}")

dict_zinnen={"iscs_zinnen":iscs_zinnen,
             "iscs_statistics_zinnen":iscs_statistics_zinnen,
             "iscs_bootstrap_zinnen":iscs_bootstrap_zinnen, 
             "ci_zinnen":ci_zinnen,
             "p_zinnen":p_zinnen, 
             "distribution_zinnen":distribution_zinnen,
             "significant_channels_zinnen":significant_channels_zinnen,
             "num_significant_channels_zinnen":num_significant_channels_zinnen,
             "p_adjusted_zinnen":p_adjusted_zinnen,
             "significant_channels_adjusted_zinnen":significant_channels_adjusted_zinnen,
             "num_significant_channels_adjusted_zinnen":num_significant_channels_adjusted_zinnen}

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1001_evoked_zinnen_block-ave.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms (fix_zinnen)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1002_evoked_zinnen_block-ave.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms (fix_zinnen)
        0 CTF compensation matrices available
        nave = 22 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Error en sub-V1003
Error en sub-V1004
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1005_evoked_zinnen_block-ave.fif ...
    Found the data of interest:
        t =   

c:\Users\UCM\anaconda3\envs\env_meg\Lib\site-packages\brainiak\utils\utils.py:996: RuntimeWarning: invalid value encountered in divide
  return numerator / denominator


iscs_statistics_zinnen, [ 2.13258647e-03  1.96933723e-03 -3.77194998e-03 -4.56188401e-03
 -3.06567945e-04  7.11437646e-03  8.79071731e-03 -1.22818397e-02
 -1.34945862e-02 -5.16523782e-05  1.25471882e-02  1.60283450e-02
  6.89933449e-03  1.67546622e-02 -2.97718870e-03  1.57946628e-02
 -1.25675479e-02 -1.19648165e-02  3.80268602e-03  4.89446878e-03
 -3.58213055e-03 -1.28096757e-02 -9.12143333e-03 -2.23034175e-02
 -9.57254038e-03 -2.32561754e-03 -8.55092718e-03 -4.87146564e-03
 -1.36299057e-02  1.00204257e-04  1.10360775e-02  1.14743118e-02
  3.02153858e-02  1.74721223e-03  9.27834638e-03  1.81299100e-02
  1.95573597e-02  1.70827991e-02  5.37358795e-03  9.63557165e-03
  1.24003922e-02  2.03811642e-02  8.80169020e-03  1.16273937e-02
  8.84028167e-03  1.33605129e-02  1.38047648e-02  1.08766833e-02
  6.21113361e-03  9.49554497e-03  9.38630014e-03  3.74510799e-03
  1.15127732e-03 -7.68746637e-04  3.83340883e-03  3.33168395e-03
  7.17281055e-03 -6.89357648e-03 -5.52655361e-03  5.21667287e-03
 

In [16]:
import pickle 
# Definir la ruta de guardado
pickle_path = ISC_path / f"dict_zinnen_{layer_script}.pkl"

# Guardar el diccionario
with open(pickle_path, 'wb') as f:
    pickle.dump(dict_zinnen, f)

print(f"Diccionario guardado en: {pickle_path}")

Diccionario guardado en: g:\MOUS_204\MOUS_visual\output_analysis\analysis_block\ISC_block\dict_zinnen_block.pkl


In [17]:
## woorden ISCS

# subjects=["sub-A2002", "sub-A2003"]
n_subjects=len(subjects)
# del stc_woorden_general
# del data_subj
# del subj
array_woorden=np.zeros((n_times,n_dipoles, n_subjects))



number_subjects=np.zeros(len(subjects))

for i in range(0,n_subjects):
    try:
        subj=subjects[i]
        
        evokeds_woorden=mne.read_evokeds(evoked_path / f"{subj}_evoked_woorden_block-ave.fif")
        ## evokeds_woorden es una lista

        ##cojo el primer elemento, solo tengo una lista
        evoked_woorden=evokeds_woorden[0]
        ####stc comes as an array of (n_dipoles, n_times)
        #take only the meg channels
        data_subj=evoked_woorden.pick("mag", exclude="bads").data
        n_dipoles=data_subj.shape[0]
        n_times=data_subj.shape[1]
        # del stc_woorden_morphed
        ##as everything on an array is of same dtype,  number of subject)
        subj_number = int(re.findall(r'\d+', subj)[0])  # Encuentra los dígitos y convierte a entero

        # Transponer los datos (n_times x n_dipoles -> n_dipoles x n_times)
        data_subj_swapped = data_subj.T

        array_woorden[:,:,i]=data_subj_swapped
        # number_subjects[i]=subj_number
        # del data_subj
    except:
        print(f"Error en {subj}")
        continue


#number_subjects_expanded = np.broadcast_to(number_subjects, (n_times,n_dipoles, n_subjects))

#array_woorden[:, :, :] = number_subjects_expanded




## brainiak need an array like (n_TRs x n_voxels x n_subjects)
#so i have to create an array
#calculus of ISC correlation

iscs_woorden= isc(data=array_woorden, pairwise=False, summary_statistic=None, tolerate_nans=True)

iscs_statistics_woorden=brainiak.isc.compute_summary_statistic(iscs_woorden, summary_statistic='mean', axis=0)
iscs_statistics_woorden.shape


##bootstrap woorden gives you the isc values, the confidence intervals, the p values and the distribution of the isc values
## so  isc_woorden=iscs_bootstrap_woorden [0]

iscs_bootstrap_woorden, ci_woorden,p_woorden, distribution_woorden= brainiak.isc.bootstrap_isc(iscs_woorden, 
pairwise=False, summary_statistic='median', n_bootstraps=1000, 
ci_percentile=95, side='right', random_state=None)


print(f"iscs_statistics_woorden, {iscs_statistics_woorden}")
print(f"iscs_bootstrap_woorden: {iscs_woorden}")
print(f"ci_woorden: {ci_woorden}")
print(f"p_woorden: {p_woorden}")
print(f"distribution_woorden: {distribution_woorden}")
print(f"iscs_statistics_woorden: {iscs_statistics_woorden}")

threshold = 0.05

# Encontrar los índices (canales) donde p_woorden[0] es menor que 0.05
significant_channels_woorden = np.where(p_woorden < threshold)[0]

# Contar cuántos canales cumplen la condición
num_significant_channels_woorden = len(significant_channels_woorden)

# Imprimir los canales significativos
print("Canales significativos woorden (p < 0.05):", significant_channels_woorden)
print("Número total de canales significativos general:", num_significant_channels_woorden)



# Aplicar la corrección de Benjamini-Hochberg (FDR)
_, p_adjusted_woorden, _, _ = multipletests(p_woorden, method='fdr_bh')
significant_channels_adjusted_woorden = np.where(p_adjusted_woorden < threshold)[0]

# Mostrar cuántos canales siguen siendo significativos
num_significant_channels_adjusted_woorden = len(significant_channels_adjusted_woorden)
print(f"Canales significativos woorden después de FDR-BH: {significant_channels_adjusted_woorden} de {len(channels_mag)}")

dict_woorden={"iscs_woorden":iscs_woorden,
             "iscs_statistics_woorden":iscs_statistics_woorden,
             "iscs_bootstrap_woorden":iscs_bootstrap_woorden, 
             "ci_woorden":ci_woorden,
             "p_woorden":p_woorden, 
             "distribution_woorden":distribution_woorden,
             "significant_channels_woorden":significant_channels_woorden,
             "num_significant_channels_woorden":num_significant_channels_woorden,
             "p_adjusted_woorden":p_adjusted_woorden,
             "significant_channels_adjusted_woorden":significant_channels_adjusted_woorden,
             "num_significant_channels_adjusted_woorden":num_significant_channels_adjusted_woorden}

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1001_evoked_woorden_block-ave.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms (fix_woorden)
        0 CTF compensation matrices available
        nave = 20 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1002_evoked_woorden_block-ave.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms (fix_woorden)
        0 CTF compensation matrices available
        nave = 22 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Error en sub-V1003
Error en sub-V1004
Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block\sub-V1005_evoked_woorden_block-ave.fif ...
    Found the data of interest:
        t

In [18]:
import pickle 
# Definir la ruta de guardado
pickle_path = ISC_path / f"dict_woorden_{layer_script}.pkl"

# Guardar el diccionario
with open(pickle_path, 'wb') as f:
    pickle.dump(dict_woorden, f)

print(f"Diccionario guardado en: {pickle_path}")

Diccionario guardado en: g:\MOUS_204\MOUS_visual\output_analysis\analysis_block\ISC_block\dict_woorden_block.pkl
